# mlops-emotion-pipeline-group-12-iit-j

**Course:** MLOps · PGD AI Program · IIT Jodhpur · **Group 12**

---

## Notebook map

This is the master training notebook.

| § | Task | Owner | Status |
|---|---|---|---|
| 1 | Environment & secrets (Kaggle) | Venkata | ⬜ |
| 2 | Data preparation & normalisation (T2) | Amit | ⬜ |
| 3 | **Select & load model from Hugging Face (T3)** | **Aishwarya** | ✅ |
| 4 | Train V1 + V2 on Kaggle, log to W&B (T4) | Venkata | ⬜ |
| 5 | Push best model to Hugging Face Hub (T5) | Venkata | ⬜ |

> Tasks T1 (repo), T6 (Docker), T7 (GitHub Actions), T8 (W&B dashboard) live outside this notebook.

---

# § 3 - Select & Load a Model from Hugging Face

**Owner:** Aishwarya Mishra (G25AIT2137)

**Model:** [`distilbert-base-uncased`](https://huggingface.co/distilbert-base-uncased)  
**Dataset:** [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) — 6-class English emotion classification (sadness, joy, love, anger, fear, surprise)

## § 3.1 - Justification (≈140 words, references the HF model card)

We chose `distilbert-base-uncased` because its [model card](https://huggingface.co/distilbert-base-uncased) describes it as a distilled version of BERT-base with **40% fewer parameters (~66M)** and **60% faster inference**, while retaining roughly **97% of BERT-base's GLUE score**. The checkpoint weighs about **268 MB**, comfortably within Kaggle's free T4 GPU budget, and trains a full epoch on the 16k-row `dair-ai/emotion` training split in a few minutes leaving plenty of GPU budget for the two hyperparameter versions required by Task 4. The uncased tokenizer is a natural fit for our cleaning pipeline (Task 2 lowercases the text), and the model exposes a standard sequence-classification head, so we can plug in `num_labels=6` from `id2label.json` without architectural changes. Together these properties give us the best accuracy-per-GPU-hour trade-off for a 6-class English emotion task.

## § 3.2 - Install dependencies

Installs the packages this section needs:

- `transformers` - tokenizer + `AutoModelForSequenceClassification`
- `torch` - forward pass for the sanity check
- `huggingface_hub` - pinned here so § 5 (model push) uses the same version

Kaggle's GPU image ships these already, so on Kaggle this is effectively a version-pin. Locally it installs fresh. `-q` keeps the output quiet.

In [1]:
%pip install -q "transformers>=4.40.0" "torch>=2.2.0" "huggingface_hub>=0.23.0"

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## § 3.3 - Load the tokenizer (3 marks)

We use `AutoTokenizer.from_pretrained` so the class is inferred from the model name. For `distilbert-base-uncased` this resolves to `DistilBertTokenizerFast` - a fast Rust-backed tokenizer that lowercases input and produces WordPiece tokens with a 30,522-token vocabulary.

The tokenizer is what turns each cleaned emotion sentence into the `input_ids` / `attention_mask` tensors the model expects.

In [2]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"class       : {tokenizer.__class__.__name__}")
print(f"vocab_size  : {tokenizer.vocab_size}")
print(f"do_lower    : {tokenizer.do_lower_case}")
print(f"max_length  : {tokenizer.model_max_length}")

/Users/slc01/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/slc01/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


class       : DistilBertTokenizerFast
vocab_size  : 30522
do_lower    : True
max_length  : 512


### Quick smoke-test on a sample emotion sentence

Confirms the tokenizer lowercases, splits with WordPiece, and produces the special tokens (`[CLS]` … `[SEP]`) the classification head expects.

In [3]:
sample = "I am feeling really hopeful about tomorrow"
encoded = tokenizer(sample, return_tensors="pt")

print("tokens   :", tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))
print("input_ids:", encoded["input_ids"].tolist())
print("mask     :", encoded["attention_mask"].tolist())

tokens   : ['[CLS]', 'i', 'am', 'feeling', 'really', 'hopeful', 'about', 'tomorrow', '[SEP]']
input_ids: [[101, 1045, 2572, 3110, 2428, 17772, 2055, 4826, 102]]
mask     : [[1, 1, 1, 1, 1, 1, 1, 1, 1]]


## § 3.4 - Load the model with `num_labels` from `id2label.json` (3 marks)

Two steps:

1. Read `id2label.json` (produced by Task 2's `prepare_data.py`) and coerce keys to `int` — JSON only supports string keys, but `transformers` expects ints.
2. Call `AutoModelForSequenceClassification.from_pretrained` with `num_labels`, `id2label`, and `label2id`. Passing all three means the saved checkpoint carries human-readable labels straight into the HF Hub push (§ 5) and the Docker inference output (T6) — no extra mapping step downstream.

Hugging Face will warn that the classification head is newly initialised; that is **expected** — the base model was pre-trained on masked-LM, and we are adding a fresh 6-class head on top, which § 4 will fine-tune.

In [4]:
import json
from pathlib import Path

# Repo root = notebook's parent's parent (notebooks/ -> repo)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ID2LABEL_PATH = REPO_ROOT / "id2label.json"

with open(ID2LABEL_PATH, "r", encoding="utf-8") as fh:
    raw = json.load(fh)

id2label = {int(k): v for k, v in raw.items()}
label2id = {v: k for k, v in id2label.items()}

print(f"id2label.json @ {ID2LABEL_PATH}")
print(f"num_labels   : {len(id2label)}")
print(f"id2label     : {id2label}")

id2label.json @ /Users/slc01/Documents/MLOps/task3_files/id2label.json
num_labels   : 6
id2label     : {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

print(f"class      : {model.__class__.__name__}")
print(f"num_labels : {model.config.num_labels}")
print(f"id2label   : {model.config.id2label}")
print(f"params     : {sum(p.numel() for p in model.parameters()):,}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


class      : DistilBertForSequenceClassification
num_labels : 6
id2label   : {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
params     : 66,958,086


### Forward-pass sanity check

We push the encoded sample through the model and verify the logits shape is `(batch=1, num_labels=6)`. The values themselves are meaningless until § 4 fine-tunes the classifier — we are only confirming the head is wired correctly.

In [6]:
import torch

model.eval()
with torch.no_grad():
    outputs = model(**encoded)

assert model.config.num_labels == 6, "expected 6 emotion labels"
assert outputs.logits.shape == (1, 6), f"unexpected logits shape: {outputs.logits.shape}"

print(f"logits.shape : {tuple(outputs.logits.shape)}")
print(f"logits       : {outputs.logits.tolist()}")
print("ok — tokenizer + model are wired correctly for § 4 to take over.")

logits.shape : (1, 6)
logits       : [[0.10816274583339691, -0.021679453551769257, -0.08530716598033905, -0.22705936431884766, -0.12134601175785065, 0.0717039629817009]]
ok — tokenizer + model are wired correctly for § 4 to take over.
